In [1]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

In [2]:
import numpy as np
import pandas as pd
import requests
import openmeteo_requests
import requests_cache
from retry_requests import retry

In [3]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 47.5788221,
	"longitude": -122.4112033,
	"hourly": ["temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature", "precipitation_probability", "precipitation", "rain", "is_day"],
	"timezone": "America/Los_Angeles",
	"past_days": 2,
	"wind_speed_unit": "mph",
	"temperature_unit": "fahrenheit",
	"precipitation_unit": "inch",
	"forecast_hours": 12,
	"past_hours": 6,
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_dew_point_2m = hourly.Variables(2).ValuesAsNumpy()
hourly_apparent_temperature = hourly.Variables(3).ValuesAsNumpy()
hourly_precipitation_probability = hourly.Variables(4).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(5).ValuesAsNumpy()
hourly_rain = hourly.Variables(6).ValuesAsNumpy()
hourly_is_day = hourly.Variables(7).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["dew_point_2m"] = hourly_dew_point_2m
hourly_data["apparent_temperature"] = hourly_apparent_temperature
hourly_data["precipitation_probability"] = hourly_precipitation_probability
hourly_data["precipitation"] = hourly_precipitation
hourly_data["rain"] = hourly_rain
hourly_data["is_day"] = hourly_is_day

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Coordinates: 47.555702209472656°N -122.38996887207031°E
Elevation: 0.0 m asl
Timezone: b'America/Los_Angeles'b'GMT-7'
Timezone difference to GMT+0: -25200s

Hourly data
                         date  temperature_2m  relative_humidity_2m  \
0  2026-03-27 08:00:00+00:00       39.400700                  84.0   
1  2026-03-27 09:00:00+00:00       42.370701                  82.0   
2  2026-03-27 10:00:00+00:00       45.880699                  76.0   
3  2026-03-27 11:00:00+00:00       48.220699                  75.0   
4  2026-03-27 12:00:00+00:00       50.020699                  72.0   
5  2026-03-27 13:00:00+00:00       50.020699                  69.0   
6  2026-03-27 14:00:00+00:00       50.110703                  68.0   
7  2026-03-27 15:00:00+00:00       50.560699                  70.0   
8  2026-03-27 16:00:00+00:00       50.560699                  68.0   
9  2026-03-27 17:00:00+00:00       49.660702                  67.0   
10 2026-03-27 18:00:00+00:00       47.590698                

In [4]:
import sys
sys.path.insert(0, "..")
from src.weather_client import WeatherClient

In [5]:
client = WeatherClient()

In [31]:
raw = client.get_forecast_raw()
print(raw)
raw['latitude']

{'latitude': 47.555702, 'longitude': -122.38997, 'generationtime_ms': 0.1665353775024414, 'utc_offset_seconds': -25200, 'timezone': 'America/Los_Angeles', 'timezone_abbreviation': 'GMT-7', 'elevation': 0.0, 'hourly_units': {'time': 'iso8601', 'temperature_2m': '°F', 'apparent_temperature': '°F', 'relativehumidity_2m': '%', 'windspeed_10m': 'mp/h', 'precipitation': 'inch', 'cloudcover': '%', 'precipitation_probability': '%', 'is_day': '', 'rain': 'inch'}, 'hourly': {'time': ['2026-03-27T08:00', '2026-03-27T09:00', '2026-03-27T10:00', '2026-03-27T11:00', '2026-03-27T12:00', '2026-03-27T13:00', '2026-03-27T14:00', '2026-03-27T15:00', '2026-03-27T16:00', '2026-03-27T17:00', '2026-03-27T18:00', '2026-03-27T19:00', '2026-03-27T20:00', '2026-03-27T21:00', '2026-03-27T22:00', '2026-03-27T23:00', '2026-03-28T00:00', '2026-03-28T01:00'], 'temperature_2m': [39.4, 42.4, 45.9, 48.2, 50.0, 51.7, 50.7, 50.5, 50.7, 50.0, 48.0, 45.7, 44.4, 43.5, 42.6, 41.8, 42.2, 42.0], 'apparent_temperature': [34.5, 3

47.555702

In [7]:
df = client.get_forecast_df()

In [8]:
df

,time,temperature_2m,apparent_temperature,relativehumidity_2m,windspeed_10m,precipitation,cloudcover,precipitation_probability,is_day,rain
0,2026-03-27 08:00:00,39.4,34.5,84,3.8,0.0,20,0,1,0.0
1,2026-03-27 09:00:00,42.4,37.2,82,5.3,0.0,51,0,1,0.0
2,2026-03-27 10:00:00,45.9,40.5,76,6.4,0.0,51,0,1,0.0
3,2026-03-27 11:00:00,48.2,42.8,75,7.4,0.0,43,0,1,0.0
4,2026-03-27 12:00:00,50.0,45.4,72,7.8,0.0,98,0,1,0.0
5,2026-03-27 13:00:00,50.0,45.0,69,9.4,0.0,100,0,1,0.0
6,2026-03-27 14:00:00,50.1,44.1,68,10.3,0.0,100,0,1,0.0
7,2026-03-27 15:00:00,50.6,45.0,70,10.4,0.0,100,0,1,0.0
8,2026-03-27 16:00:00,50.6,44.1,68,10.4,0.0,100,0,1,0.0
9,2026-03-27 17:00:00,49.7,42.8,67,10.2,0.0,100,0,1,0.0


In [15]:
for row in df.itertuples():
    print(row.time)

2026-03-27 08:00:00
2026-03-27 09:00:00
2026-03-27 10:00:00
2026-03-27 11:00:00
2026-03-27 12:00:00
2026-03-27 13:00:00
2026-03-27 14:00:00
2026-03-27 15:00:00
2026-03-27 16:00:00
2026-03-27 17:00:00
2026-03-27 18:00:00
2026-03-27 19:00:00
2026-03-27 20:00:00
2026-03-27 21:00:00
2026-03-27 22:00:00
2026-03-27 23:00:00
2026-03-28 00:00:00
2026-03-28 01:00:00


In [26]:
client.get_forecast_raw()['hourly'].values

<function dict.values>

In [36]:
import datetime
datetime.time

datetime.time

In [40]:
import time
time.time()

1774647752.292945